# Блок 1: Фундамент. Управление контекстом и воспроизводимостью 🛠
Добро пожаловать на первый блок нашего интенсива!

**Цель блока:**
1. Настроить полностью локальное и независимое окружение для AI-инженерии.
2. Изучить на практике, как параметры модели (Temperature, Seed) влияют на детерминированность кода.
3. Применить паттерн Chain of Thought (CoT) для повышения качества ответов.
4. Научиться собирать базовый контекст из внешних файлов.

Вместо платных API мы будем использовать бесплатную видеокарту Google Colab (Убедитесь, что в меню `Среда выполнения` -> `Сменить среду выполнения` выбран GPU T4) и Open-Source модель `qwen2.5-coder:3b` через **Ollama**.

In [1]:
# 1. Устанавливаем необходимые системные зависимости (zstd)
!apt-get update && apt-get install -y zstd

# 2. Устанавливаем движок Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Устанавливаем библиотеки LangChain для интеграции
!pip install -qU langchain langchain-ollama

# 4. Запускаем сервер Ollama в фоновом режиме через nohup
# Мы используем nohup и перенаправление вывода, чтобы процесс не "вешал" ячейку
import subprocess
import os
import time

print("Запускаем сервер Ollama...")
# Добавляем переменные окружения для работы с GPU в Colab
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

with open("ollama_log.txt", "w") as f:
    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=f,
        stderr=f,
        env=os.environ
    )

# Даем серверу 10 секунд на инициализацию и проверку GPU
time.sleep(10)

# Проверяем, что сервер доступен
!curl http://127.0.0.1:11434

# 5. Скачиваем модель
print("Скачиваем модель qwen2.5-coder:3b (около 1.7 ГБ)...")
!ollama pull qwen2.5-coder:3b

print("\n✅ Окружение готово! Теперь можно переходить к Заданию 1.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,797 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [39.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 k

## 1. Гиперпараметры и воспроизводимость

LLM по своей природе — вероятностные модели. При одинаковом промпте они могут выдавать разный код. В production-системах (и при отладке агентов) нам часто нужна предсказуемость.
За это отвечают два параметра:
* **Temperature (от 0.0 до 1.0):** Чем выше, тем более "креативной" (и хаотичной) становится модель. Для генерации кода мы обычно используем `0.0 - 0.2`.
* **Seed:** Числовое зерно (как в `random`). При фиксированном Seed и Temperature = 0 модель будет выдавать **строго идентичный** результат каждый раз.

Напишем базовую функцию генерации.

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

def generate_code(prompt: str, temperature: float = 0.7, seed: int = None):
    # Инициализируем модель с нужными параметрами
    llm = ChatOllama(
        model="qwen2.5-coder:3b",
        temperature=temperature,
        # Если seed не None, мы пытаемся зафиксировать генерацию
        **({"seed": seed} if seed is not None else {})
    )

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

### 📝 Задание 1
Ваша задача — доказать недетерминированность модели и затем «обуздать» её.
1. Придумайте задачу для генерации функции (например: *"Напиши python функцию генерации случайного сложного пароля"*).
2. Вызовите вашу функцию `generate_code` **3 раза** с параметрами `temperature=0.9` и `seed=None`. Распечатайте результаты и убедитесь, что код немного отличается.
3. Вызовите функцию **3 раза** с параметрами `temperature=0.0` и `seed=42`. Убедитесь, что результаты идентичны символ-в-символ.

*Место для вашего кода ниже:*

In [3]:
# TODO: Напишите код для Задания 1 здесь
prompt = "Напиши python функцию генерации случайного сложного пароля. Выведи только код, без пояснений."

<details>
<summary><b>💡 Показать решение (Задание 1)</b></summary>
  
```python
prompt = "Напиши python функцию генерации случайного сложного пароля. Выведи только код, без пояснений."

print("=== ЭКСПЕРИМЕНТ 1: Высокая температура (0.9), без Seed ===")
for i in range(3):
    res = generate_code(prompt, temperature=0.9, seed=None)
    print(f"\--- Попытка {i+1} ---\n{res[:150]}...\n") # Выводим часть, чтобы не засорять экран

print("=== ЭКСПЕРИМЕНТ 2: Нулевая температура (0.0), Seed=42 ===")
for i in range(3):
    res = generate_code(prompt, temperature=0.0, seed=42)
    print(f"\--- Попытка {i+1} ---\n{res[:150]}...\n")
```
</details>

## 2. Логика рассуждений: Chain of Thought (CoT)

Модели генерируют токены последовательно. Если попросить модель «сразу выдать ответ», она может ошибиться в сложной логике, так как у неё нет "пространства для размышлений" (скрытого состояния).
Паттерн **Chain of Thought (Цепочка мыслей)** заставляет модель писать свои рассуждения вслух, прежде чем выдать финальный артефакт. Это кардинально повышает качество кода.

Создадим dummy-файл с небольшой архитектурной задачей.

In [4]:
# Имитируем создание файла с требованиями к микросервису
%%writefile TASK.md
Задача: Написать CRUD эндпоинт на FastAPI для сущности "Task" (задача).
Ограничения:
1. Если статус задачи "DONE", ее нельзя удалить.
2. При обновлении задачи нужно проверять, что новое название не короче 5 символов.
3. Использовать Pydantic схемы.

Writing TASK.md


### 📝 Задание 2
Напишите два разных промпта:
1. **Zero-Shot (Базовый):** Просто попросите модель выполнить задачу из `TASK.md`.
2. **CoT Промпт:** Попросите модель СНАЧАЛА пошагово проанализировать ограничения и продумать логику, обернув свои мысли в теги `<thought>...</thought>`, и ТОЛЬКО ПОТОМ выдать готовый код.

Сравните, насколько более качественным и безопасным получается код при использовании CoT.

*Место для вашего кода ниже:*

In [5]:
with open("TASK.md", "r", encoding="utf-8") as f:
    task_content = f.read()

# TODO: Напишите промпты и вызовите generate_code()

prompt_basic = f"""..."""
prompt_cot = f"""..."""

<details>
<summary><b>💡 Показать решение (Задание 2)</b></summary>

```python
with open("TASK.md", "r", encoding="utf-8") as f:
    task_content = f.read()

prompt_basic = f"""
Выполни задачу:
{task_content}
Выведи готовый код.
"""

prompt_cot = f"""
Выполни задачу:
{task_content}

ИНСТРУКЦИЯ:
Перед тем как писать код, пошагово обдумай логику выполнения ограничений.
Свои рассуждения ОБЯЗАТЕЛЬНО помести внутрь тегов <thought> и </thought>.
После закрывающего тега напиши готовый python код.
"""

print("=== Без Chain of Thought ===")
print(generate_code(prompt_basic, temperature=0.1))

print("\n\n=== С применением Chain of Thought ===")
print(generate_code(prompt_cot, temperature=0.1))
```
*Обратите внимание на логи в консоли: во втором случае модель сначала "проговаривает", что ей нужны проверки `if task.status == "DONE"` и `len(new_task.title) >= 5`, и поэтому она не забывает добавить их в финальный код.*
</details>

## 3. Анатомия контекста

LLM "из коробки" ничего не знает о вашем проекте. Подготовка правильного контекстного окна (Prompt + History + RAG Data) — это 80% успеха AI-инженерии. Мы воспользуемся механизмом `PromptTemplate` из LangChain.

Создадим dummy-архитектуру нашего микросервиса.

In [6]:
%%writefile ARCHITECTURE.md
# Task Tracker Microservice
- База данных: SQLite
- ORM: SQLAlchemy 2.0
- Драйвер: aiosqlite (Асинхронный)
- Формат ответа при ошибке: {"error": "Описание ошибки", "code": HTTP_STATUS}

Writing ARCHITECTURE.md


### 📝 Задание 3
Соберите инженерный пайплайн сборки контекста, используя `PromptTemplate`.
1. Считайте файл `ARCHITECTURE.md`.
2. Создайте шаблон, который принимает переменные `{architecture_doc}` и `{user_task}`. В шаблоне укажите модели роль (например, "Ты Senior Backend Developer") и жесткое требование соблюдать архитектуру.
3. Передайте задачу `user_task = "Напиши функцию получения задачи по ID. Если её нет, верни ошибку 404."`
4. Выведите финальный код.

*Место для вашего кода ниже:*

In [7]:
from langchain_core.prompts import PromptTemplate

# TODO: Реализуйте сборку контекста и генерацию

<details>
<summary><b>💡 Показать решение (Задание 3)</b></summary>

```python
from langchain_core.prompts import PromptTemplate

# 1. Читаем архитектурный документ
with open("ARCHITECTURE.md", "r", encoding="utf-8") as f:
    arch_doc = f.read()

# 2. Создаем шаблон контекста
template = """Ты Senior Python Developer.
Твоя задача — писать код строго в соответствии с архитектурными стандартами проекта.

ТЕКУЩАЯ АРХИТЕКТУРА ПРОЕКТА:
{architecture_doc}

ЗАДАЧА ОТ ПОЛЬЗОВАТЕЛЯ:
{user_task}

Выдай только код решения, без лишних пояснений."""

prompt_template = PromptTemplate(
    input_variables=["architecture_doc", "user_task"],
    template=template
)

# 3. Формируем финальный промпт
user_task = "Напиши функцию получения задачи по ID. Если её нет, верни ошибку 404 (в нужном формате)."
final_prompt = prompt_template.format(
    architecture_doc=arch_doc,
    user_task=user_task
)

# 4. Запускаем модель
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.1)
response = llm.invoke([HumanMessage(content=final_prompt)])

print("Сформированный системный промпт:")
print(final_prompt)
print("\n--- ОТВЕТ ИИ ---")
print(response.content)
```
*Успех: Модель должна сгенерировать асинхронный код (async/await), использовать `aiosqlite` или `SQLAlchemy 2.0` методы, и вернуть ошибку в строгом формате `{"error": "...", "code": 404}`, как и требовалось в архитектуре.*
</details>

**Итоги Блока 1:**
Вы настроили локальный ИИ (не требующий интернета для генерации!), научились контролировать его "фантазию" через Temperature/Seed, улучшили логику с помощью Chain of Thought и научились "скармливать" ему файлы проекта через Context Engineering.
В следующем Блоке 2 мы автоматизируем чтение файлов и создадим настоящий векторный RAG для всего репозитория!

# Блок 2: RAG с нуля — Векторизация и Чанкинг 🗄️

**Цель блока:**
1. Создать базу знаний из файлов реального программного проекта.
2. Изучить разницу между простым и «умным» (код-ориентированным) разбиением текста (Chunking).
3. Научиться превращать текст в векторы (Embeddings) локально.
4. Реализовать семантический поиск по коду.

**Стек:**
- **Ollama** (уже установлена в Блоке 1)
- **ChromaDB** (векторное хранилище)
- **Sentence-Transformers** (локальные эмбеддинги от HuggingFace)
- **LangChain** (оркестрация)

In [8]:
# 1. Устанавливаем необходимые библиотеки
!pip install -qU chromadb sentence-transformers langchain-community langchain-huggingface

# 2. Создаем структуру нашего "подопытного" микросервиса (Task Tracker)
import os

def create_dummy_repo():
    repo_files = {
        "api.py": """
from fastapi import FastAPI, HTTPException
from models import Task
import database

app = FastAPI()

@app.post("/tasks/")
async def create_task(task: Task):
    if len(task.title) < 5:
        raise HTTPException(status_code=400, detail="Title too short")
    return database.save_task(task)

@app.get("/tasks/{task_id}")
async def read_task(task_id: int):
    return database.get_task(task_id)
        """,
        "models.py": """
from pydantic import BaseModel
from typing import Optional

class Task(BaseModel):
    id: Optional[int] = None
    title: str
    description: str
    status: str = "TODO"
        """,
        "database.py": """
import sqlite3

DB_PATH = "tasks.db"

def save_task(task):
    # Логика сохранения в SQLite
    return {"status": "saved", "id": 1}

def get_task(task_id):
    # Логика получения из базы
    return {"id": task_id, "title": "Test Task", "status": "TODO"}
        """,
        "README.md": """
# Task Tracker API
Инструкция по запуску:
1. Установите зависимости: pip install fastapi uvicorn
2. Запустите сервер: uvicorn api:app --reload
Важно: база данных инициализируется автоматически при первом запуске.
        """
    }

    os.makedirs("microservice_repo", exist_ok=True)
    for name, content in repo_files.items():
        with open(f"microservice_repo/{name}", "w", encoding="utf-8") as f:
            f.write(content.strip())
    print("✅ Dummy-репозиторий создан в папке 'microservice_repo'")

create_dummy_repo()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 

## 1. Загрузка данных и выбор стратегии Chunking

Прежде чем превратить код в векторы, его нужно нарезать на части (чанки).
*   Если нарезать слишком мелко — потеряется смысл (например, заголовок функции в одном чанке, а тело — в другом).
*   Если слишком крупно — чанк не влезет в память эмбеддинг-модели или в контекст LLM.

Для кода используется `RecursiveCharacterTextSplitter` со специальными разделителями для Python.

In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

# 1. Загружаем все файлы из репозитория
loader = DirectoryLoader("./microservice_repo", glob="**/*", loader_cls=TextLoader)
documents = loader.load()

# 2. Создаем специализированный сплиттер для Python
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=250, # специально берем маленький размер для наглядности
    chunk_overlap=20
)

# 3. Создаем обычный текстовый сплиттер для README.md
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20
)

all_splits = []
for doc in documents:
    if doc.metadata['source'].endswith(".py"):
        all_splits.extend(python_splitter.split_documents([doc]))
    else:
        all_splits.extend(text_splitter.split_documents([doc]))

print(f"Итого документов после нарезки: {len(all_splits)}")
print("Пример чанка:")
print(all_splits[0].page_content)

Итого документов после нарезки: 7
Пример чанка:
from pydantic import BaseModel
from typing import Optional

class Task(BaseModel):
    id: Optional[int] = None
    title: str
    description: str
    status: str = "TODO"


### 📝 Задание 1: Сравнение стратегий
Посмотрите на результат нарезки `all_splits`.
1. Попробуйте изменить `chunk_size` на 50 и посмотрите, как это отразится на целостности функций в `api.py`.
2. **Вопрос:** Почему для `.py` файлов мы используем `from_language(Language.PYTHON)`, а не обычный текстовый сплиттер?

## 2. Локальные Embeddings и Векторная БД

Теперь превратим текст в числа. Мы будем использовать модель `all-MiniLM-L6-v2`. Она весит всего 80Мб, работает мгновенно на CPU и отлично понимает технический английский/русский контекст.

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Инициализируем бесплатную локальную модель эмбеддингов
# Она скачается один раз из HuggingFace (доступно в РФ)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Создаем векторную базу данных Chroma в оперативной памяти
vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    collection_name="repo_collection"
)

print("✅ Векторная база знаний успешно создана!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Векторная база знаний успешно создана!


## 3. Семантический поиск vs Поиск по ключевым словам

Главная сила RAG — семантический поиск. Если вы спросите «Как сохраняются данные?», система найдет `database.save_task`, даже если в запросе нет слова "save".

In [11]:
query = "Как мне запустить этот проект?"

# Ищем 2 самых похожих чанка
docs = vectorstore.similarity_search(query, k=2)

print(f"Результаты поиска для запроса: '{query}'\n")
for i, doc in enumerate(docs):
    print(f"--- Результат #{i+1} (Источник: {doc.metadata['source']}) ---")
    print(doc.page_content)
    print("\n")

Результаты поиска для запроса: 'Как мне запустить этот проект?'

--- Результат #1 (Источник: microservice_repo/README.md) ---
# Task Tracker API
Инструкция по запуску:
1. Установите зависимости: pip install fastapi uvicorn
2. Запустите сервер: uvicorn api:app --reload
Важно: база данных инициализируется автоматически при первом запуске.


--- Результат #2 (Источник: microservice_repo/database.py) ---
def get_task(task_id):
    # Логика получения из базы
    return {"id": task_id, "title": "Test Task", "status": "TODO"}




### 📝 Задание 2: "Стресс-тест поиска"
Ваша задача — проверить качество вашей базы знаний.
1. Напишите код, который выполняет поиск по запросу: *"Какие валидации стоят на создании задачи?"*.
2. Убедитесь, что поиск нашел именно файл `api.py` и фрагмент с `if len(task.title) < 5`.
3. Попробуйте "сломать" поиск, задав очень абстрактный вопрос, и посмотрите, что он вернет.

In [12]:
# TODO: Напишите код поиска для Задания 2 здесь
query_task_2 = "Какие валидации стоят на создании задачи?"

<details>
<summary><b>💡 Показать решение (Задание 2)</b></summary>

```python
query_task_2 = "Какие валидации стоят на создании задачи?"

# Выполняем поиск
results = vectorstore.similarity_search(query_task_2, k=1)

if len(results) > 0:
    found_doc = results[0]
    print(f"Найден файл: {found_doc.metadata['source']}")
    print("Содержимое:")
    print(found_doc.page_content)
    
    if "len(task.title) < 5" in found_doc.page_content:
        print("\n✅ Успех! Система нашла правильный контекст для ответа.")
    else:
        print("\n❌ Система нашла что-то другое. Попробуйте увеличить k или изменить chunk_size.")
```
</details>

## 4. Собираем RAG-цепочку (Интеграция с LLM)

Теперь объединим всё вместе: Вопрос -> Поиск в ChromaDB -> Промпт с контекстом -> Ollama.

In [13]:
!pip install -qU langchain-classic

from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

# 1. Подключаем нашу локальную модель из Блока 1
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.1)

# 2. Создаем цепочку "Вопрос-Ответ"
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # "stuff" просто вставляет все найденные чанки в промпт
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3})
)

# 3. Задаем технический вопрос по коду
question = "Где и как в коде определен путь к базе данных SQLite?"
response = qa_chain.invoke(question)

print(f"ВОПРОС: {question}")
print("-" * 30)
print(f"ОТВЕТ ИИ:\n{response['result']}")

ВОПРОС: Где и как в коде определен путь к базе данных SQLite?
------------------------------
ОТВЕТ ИИ:
Путь к базе данных SQLite определен в переменной `DB_PATH` в файле `tasks.py`.


### Итоги Блока 2:
1. **Chunking:** Мы поняли, что код нужно резать с учетом его синтаксиса (`Language.PYTHON`).
2. **Embeddings:** Мы использовали локальную модель для превращения текста в "цифровые смыслы".
3. **ChromaDB:** Мы создали локальное хранилище, которое работает как "внешняя память" для ИИ.
4. **RAG:** Мы научили модель отвечать на вопросы по нашему проприетарному коду, который она никогда не видела в процессе обучения.

**В Блоке 3 мы пойдем дальше и создадим Агента, который сможет не только читать файлы, но и править их!**

# Блок 3: RAG с нуля — Гибридный поиск и Стресс-тест 🎯

**Цель блока:**
1. Понять ограничения векторного (семантического) поиска.
2. Подключить классический поиск по ключевым словам (алгоритм BM25).
3. Создать `EnsembleRetriever` (Гибридный поиск), объединяющий лучшее из двух подходов с помощью алгоритма Reciprocal Rank Fusion (RRF).
4. Провести автоматизированный стресс-тест базы знаний на контрольных вопросах (завершение Кейса №1).

**Стек:** LangChain, Rank-BM25.

In [14]:
# Устанавливаем библиотеку для поиска по ключевым словам (Алгоритм Okapi BM25)
# Это стандарт индустрии для полнотекстового поиска (используется под капотом ElasticSearch)
!pip install -qU rank_bm25

# Если вы перезапустили среду, раскомментируйте код ниже, чтобы пересоздать данные из Блока 2:
# (Если ячейки из Блока 2 еще в памяти, этот шаг можно пропустить)
"""
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

loader = DirectoryLoader("./microservice_repo", glob="**/*", loader_cls=TextLoader)
docs = loader.load()
python_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, chunk_size=250, chunk_overlap=20)
all_splits = python_splitter.split_documents(docs)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings, collection_name="repo_collection")
"""
print("Библиотека rank_bm25 установлена!")

Библиотека rank_bm25 установлена!


## 1. Сравнение подходов: Вектор vs Ключевые слова

*   **Векторный поиск (ChromaDB):** Отлично понимает смысл. Запрос "Как сохранить?" найдет код с функцией `insert` или `add`. Но он плохо ищет точные артикулы, версии библиотек и коды ошибок.
*   **BM25 (Ключевые слова):** Ищет точные совпадения слов и учитывает их редкость (TF-IDF). Отлично найдет `HTTPException status_code=400`, но ничего не найдет по запросу "ошибка валидации", если в коде написано `validation failed`.

Настроим BM25 поверх наших чанков кода.

In [15]:
!pip install -qU "langchain-core==0.1.12" "pydantic==1.10.15"

from langchain_community.retrievers import BM25Retriever

# 1. Инициализируем BM25 ретривер на основе наших нарезанных документов
bm25_retriever = BM25Retriever.from_documents(all_splits)

# 2. Настраиваем количество возвращаемых чанков
bm25_retriever.k = 2

# 3. Настраиваем наш старый векторный ретривер
chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ТЕСТ: Ищем точный технический термин
query = "uvicorn"

print("=== Результат семантического поиска (Chroma) ===")
for doc in chroma_retriever.invoke(query):
    print(f"[{doc.metadata.get('source', 'Unknown')}]") # Вектор может не найти точное слово, если модель не знает uvicorn

print("\n=== Результат поиска по ключевым словам (BM25) ===")
for doc in bm25_retriever.invoke(query):
    print(f"[{doc.metadata.get('source', 'Unknown')}]\n{doc.page_content[:50]}...") # BM25 найдет README.md мгновенно

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.9/218.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 1.1.1 requires langchain-core<2.0.0,>=1.2.13, but you have langchain-core 0.1.12 which is incompatible.
langchain-huggingface 1.2.1 requires langchain-core<2.0.0,>=1.2.11, but you have langchain-core 0.1.12 which is incompatible.
ollama 0.6.1 requires pydantic>=2.9, but you have pydantic 1.10.15 which is incompatible.
langchain-ollama 1.0.1 requires langchain-core<2.0.0,>=1.0.0, but you have langch

## 2. Создание Гибридного поиска (Ensemble Retriever)

Чтобы не гадать, какой поиск использовать, мы объединим их. `EnsembleRetriever` делает два запроса параллельно, а затем перевзвешивает результаты с помощью алгоритма RRF:
$score = \frac{1}{k + rank_{semantic}} + \frac{1}{k + rank_{keyword}}$

В LangChain это делается буквально в пару строк.

In [16]:
!pip install -qU langchain-classic

from langchain_classic.retrievers import EnsembleRetriever

# Создаем гибридный ретривер
# weights=[0.5, 0.5] означает, что мы доверяем обоим алгоритмам одинаково
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)

print("✅ Гибридный поиск успешно настроен!")

# Проверим, как он справится со сложным запросом
complex_query = "Где сервер валидирует длину title и возвращает 400?"
docs = ensemble_retriever.invoke(complex_query)

print(f"\nТоп-1 найденный документ для запроса '{complex_query}':")
print(f"Файл: {docs[0].metadata['source']}")
print(f"Код:\n{docs[0].page_content}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.4/344.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
google-adk 1.25.1 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
✅ Гибридный поиск успешно настроен!

Топ-1 найденный документ для запроса 'Где сервер валидирует длину title и возвращает 400?':
Файл: microservice_repo/database.py
Код:
def get_task(task_id):
    # Логика получения из базы

## 3. Практический кейс: Автоматизированный Стресс-тест

В Модуле 4 мы говорили о метриках RAG. Одной из ключевых метрик является **Context Precision (Точность контекста)** — находит ли система релевантный кусок кода (чанк) в Топ-K выдаче.

У нас есть 5 контрольных вопросов (Ground Truth) и названия файлов, в которых лежат правильные ответы. Мы напишем скрипт, который прогонит все вопросы через гибридный поиск и посчитает долю успехов.

### 📝 Задание: Разработка скрипта тестирования (Кейс №1)
Перед вами словарь `test_cases`, где ключ — это каверзный вопрос разработчика, а значение — имя файла, где находится ответ.
Ваша задача:
1. Написать цикл, который проходит по всем вопросам.
2. Для каждого вопроса вызвать `ensemble_retriever.invoke(question)`.
3. Проверить, содержится ли ожидаемое имя файла (`expected_file`) в `doc.metadata['source']` **среди Топ-3** возвращенных документов.
4. Вывести итоговую точность в процентах (Accuracy).

*Место для вашего кода ниже:*

In [17]:
test_cases = {
    "Как установить зависимости и запустить проект?": "README.md",
    "Какая таблица или файл используется для хранения БД?": "database.py",
    "Какой статус по умолчанию присваивается новой задаче?": "models.py",
    "Что произойдет, если я отправлю задачу с title равным '123'?": "api.py",
    "Какая библиотека используется для создания Pydantic схем?": "models.py"
}

# TODO: Напишите цикл оценки метрики Context Precision
total_questions = len(test_cases)
correct_retrievals = 0

# Ваш код здесь...

<details>
<summary><b>💡 Показать решение (Задание)</b></summary>

```python
test_cases = {
    "Как установить зависимости и запустить проект?": "README.md",
    "Какая таблица или файл используется для хранения БД?": "database.py",
    "Какой статус по умолчанию присваивается новой задаче?": "models.py",
    "Что произойдет, если я отправлю задачу с title равным '123'?": "api.py",
    "Какая библиотека используется для создания Pydantic схем?": "models.py"
}

total_questions = len(test_cases)
correct_retrievals = 0

print("Начинаем стресс-тест гибридного поиска...\n")

for question, expected_file in test_cases.items():
    # Получаем Топ-3 документов от гибридного ретривера (по умолчанию он вернет 4 документа: 2 от BM25 и 2 от Chroma)
    retrieved_docs = ensemble_retriever.invoke(question)[:3]
    
    # Собираем список файлов, которые попали в выдачу
    retrieved_files =[doc.metadata.get('source', '') for doc in retrieved_docs]
    
    # Проверяем, есть ли ожидаемый файл среди найденных
    # Мы используем 'in', так как source может выглядеть как 'microservice_repo/README.md'
    is_correct = any(expected_file in file_path for file_path in retrieved_files)
    
    if is_correct:
        correct_retrievals += 1
        print(f"✅ ПАС: '{question}' -> Найден в {expected_file}")
    else:
        print(f"❌ ФЕЙЛ: '{question}' -> Ожидался {expected_file}, но найдено {retrieved_files}")

# Считаем итоговую метрику
accuracy = (correct_retrievals / total_questions) * 100
print("-" * 40)
print(f"📊 Итоговая точность RAG-системы (Context Precision): {accuracy:.1f}%")

# Бонус: Сравнение
# Можете попробовать заменить ensemble_retriever на chroma_retriever
# и посмотреть, упадет ли точность.
```
</details>

## 4. Проверка боем: Ответ ИИ на сложный вопрос с использованием гибридного контекста

Теперь, когда мы уверены, что наш поиск доставляет правильные файлы, зададим вопрос LLM, используя наш обновленный ретривер.

In [18]:
!pip install -qU langchain-classic

from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

# Инициализируем модель (убедитесь, что сервер Ollama запущен)
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# Подключаем ГИБРИДНЫЙ ретривер в цепочку QA
hybrid_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever
)

complex_question = "Разработчик жалуется, что при отправке валидной задачи возникает ошибка Pydantic. Как реализована модель Task и какие поля обязательны?"

print(f"ВОПРОС: {complex_question}\n")
print("Генерация ответа ИИ...")
response = hybrid_qa_chain.invoke(complex_question)

print("-" * 40)
print(response['result'])

ВОПРОС: Разработчик жалуется, что при отправке валидной задачи возникает ошибка Pydantic. Как реализована модель Task и какие поля обязательны?

Генерация ответа ИИ...
----------------------------------------
Модель Task определена в файле models.py и содержит следующие поля:

1. `id`: Опциональный целочисленный идентификатор задачи. По умолчанию его значение равно None.
2. `title`: Обязательное строковое поле, содержащее заголовок задачи.
3. `description`: Обязательное строковое поле, содержащее описание задачи.
4. `status`: Необязательное строковое поле, содержащее статус задачи. По умолчанию его значение равно "TODO".

Ошибка Pydantic возникает, если при отправке задачи отсутствует обязательное поле `title` или `description`.


### Итоги Блока 3:
В этом блоке мы завершили **Кейс №1** программы. Вы увидели, как "инженерный" подход (добавление BM25 и перевзвешивание результатов) решает проблему "слепоты" ИИ к точным терминам и кодам. Мы также написали автоматизированный скрипт для расчета качества поиска — это фундамент тестирования (LLMOps) любых ИИ-инструментов в продакшене.

В следующем Блоке 4 мы перейдем к итерационной самокоррекции, где модели начнут общаться друг с другом для поиска багов!

# Блок 4: Итерационная самокоррекция (Паттерн "Генератор-Критик") ♻️

**Цель блока:**
1. Понять, почему ИИ не должен писать код "в один проход" (Zero-shot).
2. Реализовать паттерн разделения ролей: одна LLM пишет код, другая — ищет в нем баги.
3. Построить автоматическую петлю обратной связи (Feedback Loop).
4. Научить систему исправлять саму себя без участия человека.

**Стек:** LangChain, Python `while` loop, локальная модель Ollama (`qwen2.5-coder:3b`).

In [19]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# Инициализируем нашу локальную модель
# Мы используем низкую температуру для предсказуемости
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.1)

print("✅ Модель подключена и готова к работе.")

✅ Модель подключена и готова к работе.


## 1. Постановка задачи и "Слепой" разработчик

Представьте ситуацию: Разработчик (Генератор) получил задачу написать функцию удаления задачи из БД. Но он **не прочитал** архитектурный документ `ARCHITECTURE.md`, в котором сказано важное бизнес-правило. А вот QA-инженер (Критик) этот документ читал.

Зададим бизнес-правило:
> **Правило:** Задачу со статусом "DONE" удалять нельзя. Перед удалением (DELETE) обязательно нужно сделать SELECT и проверить статус. Если статус "DONE", нужно вернуть ошибку `{"error": "Cannot delete DONE tasks"}`.

Сначала настроим промпт Генератора.

In [20]:
# Промпт для Генератора (Он ничего не знает про статусы, пишет "в лоб")
generator_prompt_template = """
Ты Backend Разработчик.
Напиши Python-функцию (асинхронную, использует aiosqlite), которая удаляет задачу из БД по task_id.

ЗАДАЧА:
{task}

ОБРАТНАЯ СВЯЗЬ ОТ РЕВЬЮВЕРА (если есть):
{feedback}

Выведи ТОЛЬКО код. Никаких пояснений.
"""

# Промпт для Критика (Он знает правила)
critic_prompt_template = """
Ты Строгий QA/Архитектор. Твоя задача проверить написанный код.

БИЗНЕС-ПРАВИЛА:
1. Задачу со статусом "DONE" удалять нельзя.
2. Код должен сначала сделать SELECT статуса.
3. Если статус "DONE", вернуть {{"error": "Cannot delete DONE tasks"}}.

КОД ДЛЯ ПРОВЕРКИ:
{code}

ИНСТРУКЦИЯ:
Проверь, выполняет ли код бизнес-правила.
Если код ИДЕАЛЕН и содержит все проверки, напиши первой строкой слово: [APPROVED].
Если код нарушает правила, напиши первой строкой слово: [REJECTED] и ниже подробно опиши, что именно разработчик должен исправить.
"""

## 2. Проверка работы Генератора (Первая итерация)

Давайте посмотрим, какой код напишет Генератор на первой итерации, не имея обратной связи.

In [21]:
task_description = "Реализуй функцию delete_task(task_id: int)."

# Имитируем первую попытку (feedback пока пустой)
first_attempt_prompt = generator_prompt_template.format(
    task=task_description,
    feedback="Нет (это первая попытка)"
)

response_gen = llm.invoke([HumanMessage(content=first_attempt_prompt)])
initial_code = response_gen.content

print("=== КОД ОТ ГЕНЕРАТОРА (Итерация 1) ===")
print(initial_code)

=== КОД ОТ ГЕНЕРАТОРА (Итерация 1) ===
```python
import aiosqlite

async def delete_task(task_id: int):
    async with aiosqlite.connect('your_database.db') as conn:
        async with conn.cursor() as cursor:
            await cursor.execute('DELETE FROM tasks WHERE task_id = ?', (task_id,))
            await conn.commit()
```


*(Скорее всего, модель напишет простой `DELETE FROM tasks WHERE id = ?` без всяких проверок).*

### 📝 Задание для студентов: Создание Автоматического Цикла (Feedback Loop)
Теперь самое интересное. Ваша задача написать `while` цикл, который заставит эти две роли общаться друг с другом, пока Критик не напишет `[APPROVED]`.

**Алгоритм цикла (максимум 3 итерации):**
1. Отправьте `initial_code` в промпт Критика.
2. Получите ответ Критика.
3. Проверьте: если в ответе Критика есть `[APPROVED]`, остановите цикл (код готов!).
4. Если `[APPROVED]` нет (или есть `[REJECTED]`), передайте ответ Критика в качестве `feedback` обратно Генератору.
5. Получите новый код от Генератора и повторите процесс.

*Место для вашего кода ниже:*

In [22]:
max_iterations = 3
current_code = initial_code
feedback = "Нет (это первая попытка)"

print("🚀 Запускаем автономный процесс самокоррекции...\n")

for i in range(1, max_iterations + 1):
    print(f"\n--- ИТЕРАЦИЯ {i} ---")

    # TODO: Напишите логику Критика
    # 1. Сформируйте critic_prompt
    # 2. Вызовите LLM


    # TODO: Напишите логику проверки
    # Если [APPROVED], прервать цикл (break)


    # TODO: Напишите логику исправления (Генератор)
    # Если Rejected, сформируйте новый generator_prompt
    # Получите новый current_code

    pass # Удалите pass и напишите ваш код

🚀 Запускаем автономный процесс самокоррекции...


--- ИТЕРАЦИЯ 1 ---

--- ИТЕРАЦИЯ 2 ---

--- ИТЕРАЦИЯ 3 ---


<details>
<summary><b>💡 Показать решение (Задание)</b></summary>

```python
max_iterations = 3
current_code = initial_code

print("🚀 Запускаем автономный процесс самокоррекции...\n")

for i in range(1, max_iterations + 1):
    print(f"\n================ ИТЕРАЦИЯ {i} ================")
    
    # 1. Работа Критика (Code Review)
    critic_prompt = critic_prompt_template.format(code=current_code)
    critic_response = llm.invoke([HumanMessage(content=critic_prompt)]).content
    
    print(f"🕵️‍♂️ ВЕРДИКТ КРИТИКА:\n{critic_response}\n")
    
    # 2. Условие выхода
    if "[APPROVED]" in critic_response.upper():
        print("✅ ИДЕАЛЬНО! Код принят критиком.")
        break
        
    if i == max_iterations:
        print("⚠️ Достигнут лимит итераций. Код так и не принят.")
        break
        
    # 3. Работа Генератора (Исправление багов)
    print("🛠️ РАЗРАБОТЧИК (ИИ) ИСПРАВЛЯЕТ КОД...")
    gen_prompt = generator_prompt_template.format(
        task=task_description,
        feedback=critic_response # Передаем замечания критика
    )
    
    current_code = llm.invoke([HumanMessage(content=gen_prompt)]).content
    print(f"💻 НОВЫЙ КОД ОТ РАЗРАБОТЧИКА:\n{current_code}")

print("\n\n🎉 === ФИНАЛЬНЫЙ РЕЗУЛЬТАТ === 🎉")
print(current_code)
```
</details>

## 3. Разбор полетов: Что сейчас произошло?

Запустив решение, вы увидите "магию" Агентного ИИ:
1. **Итерация 1:** Модель написала глупый код. Критик "увидел" отсутствие `SELECT` и выдал `[REJECTED]: Вы не проверили статус!`.
2. **Итерация 2:** Модель, получив по шапке от себя самой же (от экземпляра Критика), извинилась (в скрытом стейте) и добавила `SELECT`. Но, возможно, забыла вернуть правильный JSON-ответ.
3. **Итерация 3:** Код доведен до идеала. Критик пишет `[APPROVED]`.

### Почему это меняет всё?
В классической разработке программист сам тестирует код и тратит на это 60% времени.
В AI-разработке (Spec-Driven Development, о котором мы говорим в Модуле 6) человек пишет **Правила (Spec / Architecture)**, а LLM-агенты в цикле `Генерация -> Тест -> Исправление` сами доводят код до рабочего состояния.

**Метрика успеха:** Надежность выдачи (Reliability) вырастает с ~50% (у zero-shot промпта) до ~85-90% при использовании паттерна Reflexion.

### Итоги Блока 4:
Вы своими руками собрали автоматическую петлю обратной связи! Этот же механизм лежит в основе таких мощных open-source инструментов как **Aider** или **OpenHands**.
В Блоке 5 мы дадим нашему агенту настоящие "руки" — научим его не просто генерировать текст, а физически читать файлы с жесткого диска и запускать `pytest` для проверки кода (Environment Feedback)!

# Блок 5: Инструментарий, Tool Calling и основы MCP 🛠️

**Цель блока:**
1. Понять, как LLM принимает решение об использовании внешней функции (Tool).
2. Создать инструменты взаимодействия с файловой системой (Чтение/Запись).
3. Привязать инструменты к локальной модели (Bind Tools).
4. Дать ИИ возможность самостоятельно запустить терминальную команду (`pytest`) и проанализировать результат.

**Контекст: Что такое MCP?**
Model Context Protocol (MCP) — это новый открытый стандарт. Вместо того чтобы хардкодить инструменты прямо в скрипте (как мы сделаем сейчас для простоты), MCP позволяет поднять отдельный защищенный микросервис с инструментами (например, доступ к БД), к которому любая LLM (Claude, Qwen, Llama) может подключиться через стандартизированный JSON-RPC интерфейс. Это как "USB-порт" для ИИ-агентов.

Для начала подготовим среду: создадим файл с багом и тесты к нему.

In [24]:
# 1. Установим pytest для наших экспериментов
!pip install -q pytest

In [25]:
# 2. Создаем файл с бизнес-логикой (с намеренной ошибкой)
%%writefile calculator.py
def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    # ОШИБКА: Разработчик случайно написал плюс вместо умножения!
    return a + b

Writing calculator.py


In [26]:
# 3. Создаем тесты
%%writefile test_calculator.py
import pytest
from calculator import add, multiply

def test_add():
    assert add(2, 2) == 4
    assert add(0, 5) == 5

def test_multiply():
    assert multiply(2, 3) == 6
    assert multiply(3, 3) == 9

Writing test_calculator.py


## 1. Создание инструментов (Tools)

Чтобы модель поняла, что у неё есть инструмент, нам нужно описать его в виде функции.
**Секрет успеха Tool Calling:** LLM не видит код вашей функции, она видит только её название, `Type Hints` (типы данных) и `Docstring` (строку документации). Чем точнее вы опишете, что делает инструмент, тем правильнее ИИ его применит.

Используем декоратор `@tool` из LangChain.

In [27]:
from langchain_core.tools import tool

@tool
def read_file(filepath: str) -> str:
    """
    Читает содержимое файла с диска.
    Используй это, чтобы посмотреть код перед его изменением.
    """
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {str(e)}"

@tool
def write_file(filepath: str, content: str) -> str:
    """
    Записывает новое содержимое в файл.
    Перезаписывает файл целиком. Используй это для сохранения исправленного кода.
    """
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Success: File {filepath} was overwritten."
    except Exception as e:
        return f"Error writing file: {str(e)}"

# Посмотрим, как LangChain преобразовал нашу функцию в JSON-схему для модели:
print("JSON-схема инструмента read_file:")
print(read_file.args_schema.schema_json(indent=2))

JSON-схема инструмента read_file:
{
  "description": "\u0427\u0438\u0442\u0430\u0435\u0442 \u0441\u043e\u0434\u0435\u0440\u0436\u0438\u043c\u043e\u0435 \u0444\u0430\u0439\u043b\u0430 \u0441 \u0434\u0438\u0441\u043a\u0430.\n\u0418\u0441\u043f\u043e\u043b\u044c\u0437\u0443\u0439 \u044d\u0442\u043e, \u0447\u0442\u043e\u0431\u044b \u043f\u043e\u0441\u043c\u043e\u0442\u0440\u0435\u0442\u044c \u043a\u043e\u0434 \u043f\u0435\u0440\u0435\u0434 \u0435\u0433\u043e \u0438\u0437\u043c\u0435\u043d\u0435\u043d\u0438\u0435\u043c.",
  "properties": {
    "filepath": {
      "title": "Filepath",
      "type": "string"
    }
  },
  "required": [
    "filepath"
  ],
  "title": "read_file",
  "type": "object"
}


/tmp/ipykernel_243/3584880384.py:30: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(read_file.args_schema.schema_json(indent=2))


## 2. Привязка инструментов к Модели (Bind Tools)

Теперь "подарим" эти инструменты нашей модели. Когда модель решит использовать инструмент, она не вернет обычный текст, она вернет специальный объект `tool_calls`.

In [28]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Инициализируем модель (температура 0 для точных вызовов)
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# Привязываем инструменты
tools =[read_file, write_file]
llm_with_tools = llm.bind_tools(tools)

# ТЕСТ: Просим модель прочитать файл
query = "Прочитай содержимое файла calculator.py и расскажи, какие функции там есть."

print("Отправляем запрос модели...")
response = llm_with_tools.invoke([HumanMessage(content=query)])

print("\n=== ОТВЕТ МОДЕЛИ ===")
if response.tool_calls:
    print(f"Модель решила вызвать инструмент(ы): {len(response.tool_calls)}")
    for tc in response.tool_calls:
        print(f"Название инструмента: {tc['name']}")
        print(f"Аргументы: {tc['args']}")
else:
    print("Модель ответила просто текстом (не использовала инструмент):")
    print(response.content)

Отправляем запрос модели...

=== ОТВЕТ МОДЕЛИ ===
Модель ответила просто текстом (не использовала инструмент):
{"name": "read_file", "arguments": {"filepath": {"type": "string", "value": "calculator.py"}}}


In [29]:
import json  # <--- Добавили импорт
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Инициализируем модель (температура 0 для точных вызовов)
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# Привязываем инструменты
tools = [read_file, write_file]
llm_with_tools = llm.bind_tools(tools)

# ТЕСТ: Просим модель прочитать файл
query = "Прочитай содержимое файла calculator.py и расскажи, какие функции там есть."

print("Отправляем запрос модели...")
response = llm_with_tools.invoke([HumanMessage(content=query)])

print("\n=== ОТВЕТ МОДЕЛИ ===")
if response.tool_calls:
    print(f"Модель решила вызвать инструмент(ы): {len(response.tool_calls)}")
    for tc in response.tool_calls:
        print(f"Название инструмента: {tc['name']}")
        print(f"Аргументы: {tc['args']}")
else:
    # === ВОТ СЮДА ВСТРОЕН НАШ ФОЛБЭК ===
    if "{" in response.content:
        try:
            # Пытаемся вытащить JSON из сырого текста
            tool_data = json.loads(response.content)
            print("⚠️ Сработал Fallback! Модель выдала JSON текстом, а не через протокол.")
            print(f"Название инструмента: {tool_data.get('name')}")
            print(f"Аргументы: {tool_data.get('arguments')}")
        except json.JSONDecodeError:
            print("Модель ответила текстом (похожим на JSON, но с ошибкой):")
            print(response.content)
    else:
        print("Модель ответила просто текстом (не использовала инструмент):")
        print(response.content)

Отправляем запрос модели...

=== ОТВЕТ МОДЕЛИ ===
⚠️ Сработал Fallback! Модель выдала JSON текстом, а не через протокол.
Название инструмента: read_file
Аргументы: {'filepath': {'type': 'string', 'value': 'calculator.py'}}


*(Вы должны увидеть, что модель сама сформировала JSON с аргументом `filepath: "calculator.py"` для вызова `read_file`)*

### 📝 Задание для студентов: Создание инструмента запуска тестов
Сейчас ИИ умеет читать и писать файлы. Но он не знает, правильно ли он исправил ошибку. Ему нужна обратная связь от окружения (Environment Feedback).

**Ваша задача:**
1. Напишите функцию `run_pytest`, оберните её декоратором `@tool`.
2. Внутри функции используйте стандартную библиотеку Python `subprocess` для запуска команды `pytest test_calculator.py -v`.
3. Функция должна возвращать строковый вывод (stdout) этой команды (чтобы ИИ смог прочитать логи падения тестов).
4. Протестируйте, как ИИ вызывает этот инструмент.

*Подсказка: Для `subprocess.run` используйте аргументы `capture_output=True` и `text=True`.*

*Место для вашего кода ниже:*

In [30]:
import subprocess
from langchain_core.tools import tool

# TODO: 1. Реализуйте инструмент run_pytest
@tool
def run_pytest(test_file: str) -> str:
    """
    Напишите качественный docstring здесь!
    Он должен объяснять ИИ, что делает этот инструмент.
    """
    pass # Ваш код здесь


# TODO: 2. Привяжите все ТРИ инструмента к модели
# llm_with_all_tools = ...

# TODO: 3. Отправьте запрос модели
# query = "Запусти тесты в файле test_calculator.py и скажи, упали ли они."
# Выведите tool_calls ответа.

<details>
<summary><b>💡 Показать решение (Задание)</b></summary>

```python
import subprocess
from langchain_core.tools import tool

# 1. Создаем инструмент
@tool
def run_pytest(test_file: str = "test_calculator.py") -> str:
    """
    Запускает юнит-тесты через pytest для указанного файла.
    Возвращает лог консоли. Используй этот инструмент, чтобы проверить код на ошибки (баги).
    """
    try:
        # Запускаем pytest и перехватываем вывод
        result = subprocess.run(
            ["pytest", test_file, "-v"],
            capture_output=True,
            text=True
        )
        # Возвращаем комбинированный вывод stdout и stderr
        return result.stdout + "\n" + result.stderr
    except Exception as e:
        return f"Failed to run tests: {str(e)}"

# 2. Привязываем инструменты
tools =[read_file, write_file, run_pytest]
llm_with_all_tools = llm.bind_tools(tools)

# 3. Тестируем решение ИИ
query = "Запусти тесты в файле test_calculator.py. Скажи мне, какие именно тесты упали?"

response = llm_with_all_tools.invoke([HumanMessage(content=query)])

print("\n=== ВЫЗОВ ИНСТРУМЕНТА ===")
if response.tool_calls:
    for tc in response.tool_calls:
        print(f"Вызван инструмент: {tc['name']}")
        print(f"Аргументы: {tc['args']}")
        
        # 4. Фактическое выполнение функции (Симуляция работы Агента)
        print("\n=== РЕЗУЛЬТАТ ВЫПОЛНЕНИЯ (Лог тестов) ===")
        # В реальном LangGraph/Agent этот шаг происходит автоматически
        tool_func = globals()[tc['name']]
        actual_result = tool_func.invoke(tc['args'])
        print(actual_result[:500] + "...\n(вывод обрезан)")
else:
    print(response.content)
```
</details>

## 3. Как Агент понимает ответ инструмента? (Механика)

Когда модель вызывает `run_pytest`, она останавливает генерацию. Программа (ваш скрипт) перехватывает `tool_calls`, физически запускает `pytest` в Ubuntu-окружении Colab, получает красный лог с ошибками и отправляет его обратно в модель специальным сообщением `ToolMessage`.

Получив этот лог, ИИ делает вывод: *"Ага, тест `test_multiply` упал, потому что ожидал 6, а получил 5. Значит, в функции сложение вместо умножения"*.

После этого ИИ вызывает инструмент `write_file`, чтобы исправить баг, и снова вызывает `run_pytest`, чтобы убедиться, что всё зеленое. **Это и есть автономный ИИ-разработчик!**

### Итоги Блока 5:
Вы успешно освоили концепцию Tool Calling.
1. Узнали, как превратить Python-код в "руки" для ИИ.
2. Поняли важность Docstrings (Документации), так как LLM принимает решения на основе текста.
3. Подготовили мощный инструмент обратной связи — запуск тестов.
4. Поняли принцип работы протокола MCP (со стандартизированным доступом к инструментам).

# Блок 6: Автономный цикл исправления кода (AI Agent) 🤖

**Цель блока:**
1. Понять архитектуру ReAct (Reason + Act) — как ИИ рассуждает и действует.
2. Научиться управлять историей сообщений, включая `ToolMessage` (ответы от инструментов).
3. Создать замкнутый цикл: `Ошибка -> Тест -> Чтение -> Исправление -> Тест (Успех)`.
4. Завершить интенсив созданием работающего прототипа автономного ИИ-инженера.

In [31]:
# 1. Быстро восстановим файлы и инструменты (на случай перезапуска ячеек)
import subprocess
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# Восстанавливаем сломанный калькулятор и тесты
with open("calculator.py", "w") as f:
    f.write("def add(a: int, b: int) -> int:\n    return a + b\n\ndef multiply(a: int, b: int) -> int:\n    return a + b # ОШИБКА ЗДЕСЬ\n")

with open("test_calculator.py", "w") as f:
    f.write("import pytest\nfrom calculator import add, multiply\n\ndef test_add():\n    assert add(2, 2) == 4\n\ndef test_multiply():\n    assert multiply(2, 3) == 6\n")

# Восстанавливаем инструменты
@tool
def read_file(filepath: str) -> str:
    """Читает содержимое файла. Возвращает текст кода."""
    try:
        with open(filepath, "r", encoding="utf-8") as f: return f.read()
    except Exception as e: return f"Error: {str(e)}"

@tool
def write_file(filepath: str, content: str) -> str:
    """Полностью перезаписывает файл новым кодом. Используй для исправления багов."""
    try:
        with open(filepath, "w", encoding="utf-8") as f: f.write(content)
        return "File updated successfully."
    except Exception as e: return f"Error: {str(e)}"

@tool
def run_pytest(test_file: str) -> str:
    """Запускает тесты. Возвращает лог (stdout/stderr)."""
    try:
        res = subprocess.run(["pytest", test_file, "-v"], capture_output=True, text=True)
        return res.stdout + "\n" + res.stderr
    except Exception as e: return f"Error: {str(e)}"

# Создаем словарь доступных инструментов для удобного вызова
tools_map = {
    "read_file": read_file,
    "write_file": write_file,
    "run_pytest": run_pytest
}

# Подключаем модель
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)
agent = llm.bind_tools(list(tools_map.values()))

print("✅ Среда, инструменты и сломанный код готовы!")

✅ Среда, инструменты и сломанный код готовы!


## 1. Механика общения с инструментами (ToolMessage)

Чтобы агент "запомнил" результат выполнения команды (например, лог консоли), мы должны отправить ему этот лог в специальном формате `ToolMessage`. Этот формат жестко связывает ID вызова инструмента и его результат.

**Как выглядит цепочка сообщений в памяти агента:**
1. `HumanMessage`: "Почини баг".
2. `AIMessage`: (Содержит `tool_calls` = `[{id: '123', name: 'run_pytest'}]`).
3. **`ToolMessage`**: (Содержит лог ошибки, `tool_call_id='123'`).
4. `AIMessage`: "Ага, понял. Теперь прочитаю файл..." (Снова `tool_calls`).

### 📝 Задание: Сборка Автономного Цикла (Кейс №2)

Это ваше финальное задание. Вам нужно написать цикл `while` или `for` (ограничим в 7-10 итераций, чтобы ИИ не застрял бесконечно).

**Алгоритм работы агента:**
1. Подготовьте список `messages`. Первым идет `SystemMessage` (системный промпт), вторым `HumanMessage` (задача: "В `test_calculator.py` падают тесты. Почини код в `calculator.py`").
2. В цикле отправляйте `messages` в модель `agent.invoke()`.
3. Добавляйте ответ модели (`AIMessage`) в конец списка `messages` (это важно для памяти!).
4. Проверяйте: если в ответе **нет** `tool_calls` — значит агент считает задачу выполненной. Делаем `break`.
5. Если `tool_calls` **есть**:
   - Пройдитесь по ним циклом.
   - Извлеките имя функции и аргументы.
   - Вызовите нужную Python-функцию из словаря `tools_map`.
   - Создайте объект `ToolMessage(content=результат_функции, tool_call_id=tc["id"])` и добавьте его в `messages`.
6. Цикл повторяется: на следующем шаге модель увидит лог выполнения и решит, что делать дальше.

*Место для вашего кода ниже:*

In [32]:
# TODO: Напишите цикл автономного агента

system_prompt = """
Ты автономный AI-разработчик. Твоя задача - чинить баги в коде.
У тебя есть инструменты:
1. run_pytest - чтобы найти падающие тесты.
2. read_file - чтобы прочитать исходный код.
3. write_file - чтобы переписать файл с исправленным кодом.

ДЕЙСТВУЙ ПО ШАГАМ:
Шаг 1. Сначала запусти тесты.
Шаг 2. Изучи логи и найди файл, где произошла ошибка.
Шаг 3. Прочитай этот файл.
Шаг 4. Напиши исправленный код и сохрани его.
Шаг 5. Снова запусти тесты, чтобы убедиться, что они проходят (зеленые).
Если тесты прошли успешно, напиши "ВСЕ ИСПРАВЛЕНО" и заверши работу.
"""

messages =[
    SystemMessage(content=system_prompt),
    HumanMessage(content="Коллеги говорят, что в test_calculator.py падают тесты. Найди баг в calculator.py и исправь его.")
]

MAX_STEPS = 7

# Ваш цикл здесь...

<details>
<summary><b>💡 Показать решение (Задание)</b></summary>

```python
MAX_STEPS = 7

print("🚀 ЗАПУСК АВТОНОМНОГО АГЕНТА...\n" + "="*40)

for step in range(MAX_STEPS):
    print(f"\n--- ШАГ {step + 1} ---")
    
    # 1. Агент думает и принимает решение
    response = agent.invoke(messages)
    
    # 2. Обязательно сохраняем ответ агента в историю (память)
    messages.append(response)
    
    # 3. Если агент не вызывает инструменты, значит он закончил
    if not response.tool_calls:
        print("🤖 Агент завершил работу. Ответ:")
        print(response.content)
        break
        
    # 4. Обрабатываем вызовы инструментов
    print("🛠️ Агент решил использовать инструменты:")
    for tc in response.tool_calls:
        func_name = tc["name"]
        func_args = tc["args"]
        call_id = tc["id"]
        
        print(f" -> Вызов: {func_name}({func_args})")
        
        # Запускаем реальную функцию
        if func_name in tools_map:
            tool_result = tools_map[func_name].invoke(func_args)
        else:
            tool_result = f"Error: Tool {func_name} not found."
            
        # Обрезаем лог, чтобы не перегрузить контекст модели (опционально)
        if len(str(tool_result)) > 2000:
            tool_result = str(tool_result)[:2000] + "... [ОБРЕЗАНО]"
            
        print(" -> Результат получен (отправляем обратно агенту).")
        
        # 5. Сохраняем результат в формате ToolMessage, привязывая к ID вызова
        tool_msg = ToolMessage(
            content=str(tool_result),
            tool_call_id=call_id
        )
        messages.append(tool_msg)
        
    if step == MAX_STEPS - 1:
        print("⚠️ Достигнут лимит шагов. Агент остановлен принудительно.")
```
</details>

### [Ячейка 7: Текст]
## Разбор полетов: Как мыслил ваш Агент?

Если вы запустили решение, вы увидели магию в консоли:
1. **Шаг 1:** Агент вызвал `run_pytest({'test_file': 'test_calculator.py'})`.
2. **Шаг 2:** Получив красный лог `AssertionError: assert 5 == 6`, агент вызвал `read_file({'filepath': 'calculator.py'})`.
3. **Шаг 3:** Прочитав код, агент понял ошибку (в функции `multiply` стояло сложение) и вызвал `write_file` с правильным кодом.
4. **Шаг 4:** Агент *снова* вызвал `run_pytest`, чтобы убедиться, что всё работает.
5. **Шаг 5:** Получив лог `2 passed`, агент написал "ВСЕ ИСПРАВЛЕНО" текстом и цикл `break`нулся.

Поздравляю! Вы только что с нуля написали ядро системы **Spec-Driven Development / AI-DevOps**. Именно так работают современные Enterprise ИИ-инструменты "под капотом".

### 🎉 Итоги:
* **Блок 1:** Укротили хаос LLM с помощью Temperature, Seed и Chain-of-Thought. Запустили локальный ИИ в Colab.
* **Блок 2 и 3:** Разработали RAG-систему (VectorDB + Embeddings). Сделали гибридный поиск и написали скрипт для расчета метрики *Context Precision* на "грязных" данных.
* **Блок 4:** Реализовали паттерн *Reflexion* (Генератор-Критик), где ИИ проверяет сам себя без человека.
* **Блок 5 и 6:** Дали ИИ "руки" через *Tool Calling* (основы стандарта MCP), и собрали автономного ИИ-агента, способного фиксить баги в реальном коде.

Спасибо за участие! Вы можете сохранить этот Colab-ноутбук к себе на Google Drive (Файл -> Сохранить копию на Drive) и использовать код как шаблон для ваших рабочих проектов.